# 🔬 Employee Attrition Analysis Pipeline
**Methodology:** CRISP-DM adapted for HR Analytics  
**Dataset:** IBM HR Analytics (1,470 employees)  
**Reference:** Sec 8.2–8.4, 10, 13 of Project Report

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, f1_score
import shap
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline

In [ ]:
# 📥 Load Data (Sec 8.1)
df = pd.read_csv('../data/WA_Fn-UseC_-HR-Employee-Attrition.csv')
print(f"Dataset loaded: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

In [ ]:
# 🧹 Data Cleaning (Sec 8.2)
drop_cols = ['EmployeeCount', 'Over18', 'StandardHours', 'EmployeeNumber']
df_clean = df.drop(columns=drop_cols)
df_clean['Attrition'] = df_clean['Attrition'].map({'Yes': 1, 'No': 0})
df_clean['MonthlyIncome'] = df_clean['MonthlyIncome'].astype(float)
print("✅ Cleaning complete")

In [ ]:
# 🛠 Feature Engineering (Sec 8.3)
def assign_age_group(age):
    if age <= 25: return 'Early Career'
    elif age <= 35: return 'Core Contributor'
    elif age <= 45: return 'Senior/Management'
    else: return 'Executive/Late Career'

df_clean['AgeGroup'] = df_clean['Age'].apply(assign_age_group)
df_clean['PromotionGap'] = df_clean['YearsAtCompany'] - df_clean['YearsSinceLastPromotion']
df_clean['IncomeBelowDeptAvg'] = df_clean.groupby('Department')['MonthlyIncome'].transform('mean') > df_clean['MonthlyIncome']
df_clean['HighRiskFlag'] = (df_clean['OverTime'] == 'Yes') & (df_clean['YearsInCurrentRole'] < 2) & df_clean['IncomeBelowDeptAvg']
print("✅ Features engineered")

In [ ]:
# 📊 Key Metrics (Sec 13)
attrition_rate = df_clean['Attrition'].mean() * 100
ot_rate = df_clean[df_clean['OverTime']=='Yes']['Attrition'].mean()
no_ot_rate = df_clean[df_clean['OverTime']=='No']['Attrition'].mean()
print(f"🔹 Overall Attrition: {attrition_rate:.1f}%")
print(f"🔹 OT Attrition Multiplier: {(ot_rate/no_ot_rate):.2f}x")

In [ ]:
# 🤖 ML Pipeline (Sec 8.4, 10)
df_ml = pd.get_dummies(df_clean.drop(columns=['Attrition']), drop_first=True)
y = df_clean['Attrition']
X_train, X_test, y_train, y_test = train_test_split(df_ml, y, test_size=0.2, random_state=42, stratify=y)

rf = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)
y_prob = rf.predict_proba(X_test)[:, 1]

print("📋 Classification Report:\n", classification_report(y_test, y_pred))
print(f"📈 ROC-AUC: {roc_auc_score(y_test, y_prob):.3f} | F1: {f1_score(y_test, y_pred):.3f}")

In [ ]:
# 🔍 SHAP Explainability (Sec 10)
explainer = shap.TreeExplainer(rf)
shap_values = explainer.shap_values(X_test)
shap.summary_plot(shap_values[1], X_test, show=True, title="SHAP Feature Importance for Attrition")

In [ ]:
# 🚨 High-Risk Cohort (Sec 9, 16)
high_risk = df_clean[df_clean['HighRiskFlag']==True]
print(f"🔍 High-Risk Cohort: {len(high_risk)} employees ({len(high_risk)/len(df_clean)*100:.1f}% of headcount)")
print(f"📊 Attrition in Cohort: {high_risk['Attrition'].mean()*100:.1f}%")
print("\n🔹 Top Departments:", high_risk['Department'].value_counts().head(3).to_dict())